# Fase 2: Ajuste de Comportamiento Pedagógico (Fine-Tuning)
### Tutor Analítico Híbrido (RAG + Fine-Tuning)

En este cuaderno implementamos el proceso de **Instruction Fine-Tuning** utilizando la técnica de **QLoRA** (cuantización a 4 bits + adaptadores LoRA). El objetivo es modular el comportamiento del modelo base (ej. Llama 3 o Llama 3.1) para alinearlo a las directrices de nuestro Tutor (citas de fuentes reales, neutralidad, método socrático y mitigación de alucinaciones).

In [1]:
# 1. Instalar dependencias para el Fine-Tuning y cuantización
!pip install transformers peft trl bitsandbytes datasets torch -q

### 2. Configurar Entorno, Importar Librerías y Cargar Modelo/Tokenizador
Configuramos la compatibilidad de Windows, controlamos la carga del modelo (gated vs un-gated) y acoplamos los adaptadores LoRA.

In [2]:
import os
import locale
locale.getpreferredencoding = lambda *args, **kwargs: 'utf-8'
locale.getencoding = lambda *args, **kwargs: 'utf-8'

# --- MONKEYPATCH PARA EVITAR UNICODEDECODEERROR EN WINDOWS CON TRL ---
import builtins
import pathlib

original_open = builtins.open
def patched_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode:
        if 'encoding' in kwargs:
            if kwargs['encoding'] is None:
                kwargs['encoding'] = 'utf-8'
        elif len(args) < 4:
            kwargs['encoding'] = 'utf-8'
        elif len(args) >= 4 and args[3] is None:
            args_list = list(args)
            args_list[3] = 'utf-8'
            args = tuple(args_list)
    return original_open(*args, **kwargs)
builtins.open = patched_open

original_read_text = pathlib.Path.read_text
def patched_read_text(self, encoding=None, errors=None, newline=None):
    return original_read_text(self, encoding=encoding or 'utf-8', errors=errors, newline=newline)
pathlib.Path.read_text = patched_read_text

if hasattr(pathlib, 'PathBase'):
    original_read_text_base = pathlib.PathBase.read_text
    def patched_read_text_base(self, encoding=None, errors=None, newline=None):
        return original_read_text_base(self, encoding=encoding or 'utf-8', errors=errors, newline=newline)
    pathlib.PathBase.read_text = patched_read_text_base
# --------------------------------------------------------------------

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# MODELO BASE: Llama 3.2 1B (Opción recomendada por el profesor para tus 4GB VRAM)
# Usamos la versión de "unsloth" que descarga directo sin necesidad de token de Hugging Face
model_name = "unsloth/Llama-3.2-1B-Instruct"

# 1. Configuración de BitsAndBytes (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print(f"Cargando tokenizador y modelo base: {model_name}...")
# 2. Cargar Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configuración oficial de padding recomendada para la familia Llama 3/3.1/3.2 en bell-tuning/fine-tuning
tokenizer.pad_token = "<|finetune_right_pad_id|>"
tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids("<|finetune_right_pad_id|>")
tokenizer.padding_side = "right"

# 3. Cargar Modelo Base
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},  # Forzamos carga directa en tu GPU NVIDIA (GPU 0)
    torch_dtype=torch.bfloat16 
)

# Preparar modelo para k-bit training
model = prepare_model_for_kbit_training(model)

# 4. Configuración LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("¡Modelo cargado y adaptadores LoRA acoplados con éxito!")


c:\Users\crist\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando tokenizador y modelo base: unsloth/Llama-3.2-1B-Instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:01<00:00, 108.70it/s]


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039
¡Modelo cargado y adaptadores LoRA acoplados con éxito!


### 3. Cargar y Analizar el Dataset de Entrenamiento
Cargamos el archivo conversacional JSONL `data/dataset/tutor_seguridad_finetuning.jsonl` y aplicamos la plantilla oficial del chat nativo del modelo para mapear correctamente los roles conversacionales.

In [3]:
import json
from datasets import Dataset

dataset_path = 'data/dataset/tutor_seguridad_finetuning.jsonl'

def load_dataset_jsonl(filepath):
    examples = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                examples.append(json.loads(line))
    return examples

data_list = load_dataset_jsonl(dataset_path)
print(f"Total de ejemplos cargados: {len(data_list)}")

# Convertir al formato de HuggingFace Dataset
hf_dataset = Dataset.from_list(data_list)

def format_conversation(example):
    """Utiliza el Chat Template oficial del tokenizador para asegurar el mapeo inequívoco de tokens especiales de control."""
    formatted_text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': formatted_text}

formatted_dataset = hf_dataset.map(format_conversation)
print("\nEjemplo de prompt oficial formateado para el entrenamiento:")
print(formatted_dataset[0]['text'][:800])

Total de ejemplos cargados: 60


Map: 100%|██████████| 60/60 [00:00<00:00, 2318.00 examples/s]


Ejemplo de prompt oficial formateado para el entrenamiento:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 31 May 2026

Eres un Tutor Analítico especializado en seguridad pública y violencia en México. Tu comportamiento debe ser estrictamente pedagógico, académico y riguroso. Reglas fundamentales: (1) Cita siempre las fuentes del corpus al final de cada respuesta usando el formato [Documento X, Pág. Y]. (2) Mantén neutralidad y objetividad ante temas sensibles. (3) Si el corpus no contiene información suficiente, responde explícitamente que no puedes confirmar ese dato. (4) Usa el método socrático para guiar al usuario en análisis complejos. (5) Nunca inventes datos, cifras o fuentes.<|eot_id|><|start_header_id|>user<|end_header_id|>

¿Cuáles son las tres entidades federativas con mayo


### 4. Definición de Hiperparámetros y Entrenamiento (Optimizado para VRAM)
Establecemos los hiperparámetros en el `SFTTrainer` incluyendo mitigaciones para evitar errores de falta de memoria (OOM) en hardware local.

In [ ]:
import time
import torch
from trl import SFTConfig, SFTTrainer
from transformers import TrainerCallback

# 1. Definir un Callback de Enfriamiento para laptops
class LaptopCoolingCallback(TrainerCallback):
    def __init__(self, sleep_time=10, step_interval=4):
        """
        Pausa el entrenamiento periódicamente para permitir que la laptop se enfríe.
        - sleep_time: Segundos a pausar.
        - step_interval: Cada cuántos pasos de entrenamiento pausar.
        """
        self.sleep_time = sleep_time
        self.step_interval = step_interval

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step > 0 and state.global_step % self.step_interval == 0:
            print(f"\n❄️ [LaptopCoolingCallback] Pausando {self.sleep_time}s para enfriar la GPU/CPU (Paso {state.global_step}/24)...")
            time.sleep(self.sleep_time)


#  SISTEMA AUTO-DETECTABLE DE PRECISIÓN Y RESOLUCIÓN DE BFLOAT16

use_bf16 = False
use_fp16 = True

if torch.cuda.is_bf16_supported():
    print(" ¡Excelente! Tu GPU soporta BFloat16 de forma nativa.")
    print("Usaremos bf16=True para entrenar de forma óptima sin necesidad de GradScaler.")
    use_bf16 = True
    use_fp16 = False
else:
    print(" Tu GPU no soporta BFloat16 nativo (ej. GTX 1650). Usaremos fp16=True.")
    print("Convirtiendo los módulos de adaptadores LoRA a Float16 real...")
    
    # 1. Convertir los módulos LoRA a Float16 a nivel de clase
    for name, module in model.named_modules():
        if "lora_" in name:
            module.to(torch.float16)
            
    # 2. Forzar cualquier otro parámetro residual a Float16
    for param in model.parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)
            if param.grad is not None:
                param.grad.data = param.grad.data.to(torch.float16)
    print("¡Conversión forzada a Float16 completada con éxito!")
# =====================================================================

# 2. Definir la configuración usando SFTConfig con la precisión óptima
training_args = SFTConfig(
    output_dir="./tutor_seguridad_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=1,            # Optimización máxima de VRAM local
    gradient_accumulation_steps=8,            # Batch size efectivo de 8 (1 * 8)
    warmup_steps=2,                           # Calentamiento estable y sin advertencias
    learning_rate=2e-4,
    fp16=use_fp16,                            # Configuración dinámica
    bf16=use_bf16,                            # Configuración dinámica
    gradient_checkpointing=True,              # Crucial para evitar OOM (Out of Memory)
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=1,                          # Visualización paso a paso
    save_strategy="epoch",
    save_total_limit=1,                       # Limita el guardado a 1 solo checkpoint
    optim="paged_adamw_8bit",                 # AdamW de 8 bits paginado para liberar VRAM
    report_to="none",
    
    # --- Parámetros específicos de SFT optimizados ---
    dataset_text_field="text",
    max_length=512,                           # Reducido a 512 (Ahorro del 4x de memoria de atención)
)

# 3. Inicializar el Trainer de manera compatible
try:
    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_dataset,
        processing_class=tokenizer,           # Estándar de la versión más reciente de TRL
        args=training_args,
        callbacks=[LaptopCoolingCallback(sleep_time=10, step_interval=4)]  # Inyección del callback
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_dataset,
        tokenizer=tokenizer,
        args=training_args,
        callbacks=[LaptopCoolingCallback(sleep_time=10, step_interval=4)]
    )

print("Iniciando entrenamiento del modelo con enfriamiento activo...")
# Ejecutar entrenamiento
trainer.train()

print("\n💾 Guardando adaptadores LoRA entrenados...")
model.save_pretrained("./lora_tutor_seguridad")
tokenizer.save_pretrained("./lora_tutor_seguridad")
print("¡Entrenamiento y guardado completados con éxito!")



🚀 ¡Excelente! Tu GPU soporta BFloat16 de forma nativa.
Usaremos bf16=True para entrenar de forma óptima sin necesidad de GradScaler.


Tokenizing train dataset: 100%|██████████| 60/60 [00:00<00:00, 467.20 examples/s]


Iniciando entrenamiento del modelo con enfriamiento activo...


Step,Training Loss
1,2.851531
2,2.918601
3,2.599780
4,2.356000
5,2.090296
6,1.983192
7,1.831367
8,1.687879
9,1.531399
10,1.579486



❄️ [LaptopCoolingCallback] Pausando 10s para enfriar la GPU/CPU (Paso 4/24)...

❄️ [LaptopCoolingCallback] Pausando 10s para enfriar la GPU/CPU (Paso 8/24)...

❄️ [LaptopCoolingCallback] Pausando 10s para enfriar la GPU/CPU (Paso 12/24)...

❄️ [LaptopCoolingCallback] Pausando 10s para enfriar la GPU/CPU (Paso 16/24)...

❄️ [LaptopCoolingCallback] Pausando 10s para enfriar la GPU/CPU (Paso 20/24)...

❄️ [LaptopCoolingCallback] Pausando 10s para enfriar la GPU/CPU (Paso 24/24)...

💾 Guardando adaptadores LoRA entrenados...
¡Entrenamiento y guardado completados con éxito!
